# ASISTENTE VACACIONAL CON RAG Y TOOLS

## Imports, definición de modelos, creación de cliente

In [59]:
import os, getpass
from dotenv import load_dotenv
from google import genai
from pathlib import Path
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import faiss
from google.genai import types
from typing import Any
import requests


# cargamos variables desde .env 
dotenv_loaded = load_dotenv()

# solicita claves si no están definidas
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Introduce la API de Google: ")

# error si después de todo no tenemos API KEYs
if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("Claves no encontradas.")

print("Claves cargadas.")

Claves cargadas.


Cargamos la API key de Google en `GOOGLE_API_KEY`

In [60]:
GENERATION_MODEL = os.getenv("GOOGLE_GENERATION_MODEL", "gemini-3.5-flash-lite")
EMBEDDING_MODEL = os.getenv("GOOGLE_EMBEDDING_MODEL", "gemini-embedding-001")

print("Modelo de generación: ", GENERATION_MODEL)
print("Modelo de embeddings: ", EMBEDDING_MODEL)

Modelo de generación:  gemini-3.5-flash-lite
Modelo de embeddings:  gemini-embedding-001


In [61]:
client = genai.Client()

## Carga PDF RAG

In [62]:
PDF_PATH = Path("data/TENERIFE.pdf")

pages = []

# leemos cada página del pdf
with pymupdf.open(PDF_PATH) as doc:
    # page representa el contenido de la propia hoja, i representa el número de página
    for i, page in enumerate(doc, start=1):
        pages.append(
            {
                "page": i,
                "content": page.get_text()
            }
        )

print("Número de páginas cargadas:", len(pages))
print("Primeros caracteres:\n")
print(pages[0]["content"][:300])


Número de páginas cargadas: 25
Primeros caracteres:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- 
Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- 
Caminar por la Avenida Marítima hasta Pl


## Chunking

In [63]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", ", ", " ", ""]
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["content"])

    for chunk in page_chunks:
        chunks.append(
            {
                "chunk_id": len(chunks),
                "page": page["page"],
                "text": chunk
            }
        )

print(f"Se han obtenido {len(chunks)} chunks.")
print()
print("Chunk 1:")
print(chunks[0])

Se han obtenido 37 chunks.

Chunk 1:
{'chunk_id': 0, 'page': 1, 'text': 'TENERIFE – LUGARES DE INTERÉS \nSITIOS QUE VER \n \nZONA NORTE \n \n• Santa Cruz de Tenerife: \nSanta Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa \nCruz sería: \n- \nAparcar en el aparcamiento del Parque Marítimo (ubicación). \n- \nCaminar por la Avenida Marítima hasta Plaza de España (ubicación). \n- \nPor el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). \n- \nUna vez llegados a Plaza España, callejear un poco (subir la Calle Castillo \ndirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - \nubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del \nPríncipe - ubicación). \n-'}


Para el proceso de chunking se utliza la función recursiva de langchain `RecursiveCharacterTextSplitter`. Se ha elegido porque se ha considerado la opción más óptima de hacer el chunking. Sigue una jerarquía de separadores para dividir los chunks, ordenados de mayor a menor importancia, estipulada en el parámetro `separators`.

## Embedding 

In [64]:
def embed_texts(texts: list[str], task_type: str) -> np.array:
    """Embed lista y devuelve matriz numpy. Utilizaremos dos tipos de tarea: 
        - RETRIEVAL_DOCUMENT: para los chunks del PDF.
        - RETRIEVAL_QUERY: para las preguntas del usuario."""
    # llamamos al modelo de embedding utilizando el cliente
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type=task_type
        )
    )
    # guardamos vectores de embeddings 
    vectors = [item.values for item in response.embeddings]
    return np.array(vectors, dtype=np.float32)


In [65]:
# guardamos solo texto de todos los chunks
chunk_texts = [chunk["text"] for chunk in chunks]

# Llamamos a función de embedding
chunk_embeddings = embed_texts(
    texts=chunk_texts,
    task_type="RETRIEVAL_DOCUMENT"
)


# normalizamos vectores para similitud coseno
faiss.normalize_L2(chunk_embeddings)

# obtenemos dimensiones de ebeddings
dimension = chunk_embeddings.shape[1]

# creamos indice FAISS
index = faiss.IndexFlatIP(dimension)

# añadimos embeddings
index.add(chunk_embeddings)

print(f"Vectores almacenados: {index.ntotal}")


Vectores almacenados: 37


`faiss.normalize_L2`: convierte los embeddings en vectores de longitud 1.

`IndexFlatIP`: crea un índice que utiliza el producto interno y realiza una búsqueda exacta.

`index.add`: incorpora los embeddings al índice.

## Recuperación del contexto

In [66]:
def search_internal_docs(query: str, index, k: int = 4) -> list[dict[str, Any]]:
    """Recupera k chunks más relevantes"""
    # embedding de la query
    query_embedding = embed_texts(
        texts=[query],
        task_type="RETRIEVAL_QUERY"
    )

    # normalizamos 
    faiss.normalize_L2(query_embedding)

    # buscamos k vectores similares 
    scores, indexes = index.search(
        x=query_embedding,
        k=min(k, index.ntotal)
    )

    results = []
    # index.search() devuelve dos matrices, puede recibir más de una pregunta en la misma llamada, en [0] está la primera pregunta
    for rank, (score, idx) in enumerate (zip(scores[0], indexes[0]), start=1):
        chunk = chunks[int(idx)]
        results.append(
            {
                "rank": rank,
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "page": chunk["page"],
                "text": chunk["text"]
            }
        )

    return results

def print_recovered_chunks(results) -> None:
    for result in results:
        print("Chunk rank: ", result["rank"])
        print("Score: ", result["score"])
        print("chunk_id: ", result["chunk_id"])
        print("Page: ", result["page"])
        print("Content:")
        print(result["text"])
        print()
        print()

In [67]:
# prueba de función de recuperación de contexto 
"""results_1 = search_internal_docs(
    query="¿Qué miradores se recomiendan para visitar el Parque Rural de Anaga?",
    index=index,
    k=3
)

print_recovered_chunks(results_1)"""

'results_1 = search_internal_docs(\n    query="¿Qué miradores se recomiendan para visitar el Parque Rural de Anaga?",\n    index=index,\n    k=3\n)\n\nprint_recovered_chunks(results_1)'

In [68]:
"""results_2 = search_internal_docs(
    query="¿Dónde puedo disfrutar de la gastronomía tradicional canaria en un restaurante auténtico?",
    index=index,
    k=3
)

print_recovered_chunks(results_2)"""

'results_2 = search_internal_docs(\n    query="¿Dónde puedo disfrutar de la gastronomía tradicional canaria en un restaurante auténtico?",\n    index=index,\n    k=3\n)\n\nprint_recovered_chunks(results_2)'

In [69]:
# función para formatear contexto guardando fuente
def format_results_with_sources(results) -> str:
    blocks = []
    for i, res in enumerate(results, start=1):
        score = res.get("score")
        page = res.get("page", "?")
        chunk_id = res.get("chunk_id", "?")
        text = res.get("text")
        blocks.append(
            f"Score {score:.3f} Página {page}, chunk {chunk_id}\n{text}"
        )
    return "\n\n".join(blocks)
    

## Log para llamadas a herramientas 

In [70]:
tool_logs = []

def register_tool_call(name, arguments, result, error = None) -> None:
    tool_logs.append({
        "tool": name,
        "arguments": arguments,
        "success": error is None,
        "result": result,
        "error": str(error) if error else None
    })

## RAG como herramienta 

In [71]:
def search_internal_docs_tool(query: str, k: int = 3) -> str:
    """Busca información turística en la guía de Tenerife y devuelve los k fragmentos más relevantes con sus páginas."""
    try:
        results = search_internal_docs(query, index, k)
        register_tool_call(
            name="search_internal_docs_tool",
            arguments={
                "query": query,
                "k": k
            },
            result=results
        )
        return format_results_with_sources(results)
    except Exception as exc:
        register_tool_call(
            name="search_internal_docs_tool",
            arguments={
                "query": query,
                "k": k
            },
            result=None,
            error=exc
        )
        return f"ERROR: {str(exc)}"

## Herramienta tiempo en Tenerife

In [72]:
def get_weather_tool(date: str) -> dict[str, Any]:
    """Obtiene el tiempo para una fecha (aaaa-mm-dd) en Santa Cruz de Tenerife"""
    
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 28.4636,
        "longitude": -16.2518,
        "start_date": date,
        "end_date": date,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "weather_code"
        ],
        "timezone": "Atlantic/Canary"
    }

    try:
        response = requests.get(url, params)
        response.raise_for_status()
        data = response.json()

        weather_codes = {
            0: "Clear sky",
            1: "Mainly clear",
            2: "Partly cloudy",
            3: "Overcast",
            45: "Fog",
            61: "Light rain",
            63: "Moderate rain",
            65: "Heavy rain",
            95: "Thunderstorm"
        }

        code = data["daily"]["weather_code"][0]
        weather = weather_codes.get(code, "unknown")
        

        result = {
            "time": data["daily"]["time"][0],
            "temperature_max": data["daily"]["temperature_2m_max"][0],
            "temperature_min": data["daily"]["temperature_2m_max"][0],
            "precipitation_sum": data["daily"]["precipitation_sum"][0],
            "weather": weather
        }

        register_tool_call(
            name="get_weather_tool",
            arguments={
                "fecha": date
            },
            result=result, 
            error=None
        )
        return result
        
    except Exception as exc:
        register_tool_call(
            name="get_weather_tool",
            arguments={
                "fecha": date
            },
            result=None,
            error=exc
        )
        return f"ERROR: {str(exc)}"
    

In [73]:
# print(get_weather_tool("2026-10-20"))

## Instrucciones para los distintos modelos

In [74]:
MAIN_LLM_SYSTEM_INSTRUCTIONS = """
    Eres un asistente turístico especializado en Tenerife.
    Tu objetivo es ayudar al usuario a organizar su viaje, recomendar lugares
    de interés y resolver dudas turísticas de manera clara, útil, breve y cercana.
    
    USO DE HERRAMIENTAS:
    - Dispones de dos herramientas, una para consultar una guía turística de Tenerife
      y otra para obtener predicciones meteorológicas.
    - Utiliza search_internal_docs_tool cuando el usuario pregunte por playas,
      monumentos, restaurantes, actividades, rutas o recomendaciones turísticas.
    - Basa tus recomendaciones en la información recuperada de la guía.
    - Utiliza get_weather_tool cuando el usuario pregunte por el tiempo , las
      temperaturas, la lluvia o las condiciones meteorológicas de una fecha 
      concreta.
    - Pídele al usuario que te dé la fecha/fechas exactas sobre las que quiere la
      previsión meteorológica antes de llamar a la herramienta.
    - Si una recomendación depende del tiempo, utiliza ambas herramientas
      para combinar la información turística con la meteorológica.
    - No inventes datos meteorológicos ni afirmes haber consultado una
      herramienta si no la has ejecutado.
    
    GESTIÓN DE INFORMACIÓN:
    - Prioriza la información obtenida de las herramientas.
    - Cita las páginas de la guía turística utilizadas en tus respuestas.
    - Si la guía no contiene la información solicitada, indícalo claramente.
    - Si una herramienta devuelve un error, explícalo sin inventar resultados.
    - No presentes como actuales datos que puedan haber cambiado, como
      horarios, precios, restricciones de acceso o disponibilidad.
    
    CONTEXTO CONVERSACIONAL:
    - Ten en cuenta el historial proporcionado para mantener la coherencia.
    - Recuerda las preferencias, fechas y lugares mencionados anteriormente.
    - Interpreta preguntas de seguimiento teniendo en cuenta la conversación.
    - No vuelvas a consultar herramientas innecesariamente si la información
      anterior sigue siendo suficiente y válida.
    
    ESTILO DE RESPUESTA:
    - Responde siempre en español.
    - Utiliza un lenguaje natural, cercano y fácil de entender.
    - Ofrece respuestas concretas, evitando explicaciones innecesarias.
    - Organiza las recomendaciones de forma clara cuando sea útil.
    - Si no dispones de información suficiente, reconócelo.
"""

SUMMARIZE_LLM_SYSTEM_INSTRUCTIONS = """
    Eres un asistente especializado en resumir historiales de conversación.
    Tu tarea es condensar la conversación proporcionada para que otro
    asistente, especializado en ser una guía turística, pueda continuarla sin perder 
    información relevante.
    Instrucciones:
    - Conserva las preferencias, necesidades y restricciones del usuario.
    - Conserva los lugares turísticos, fechas y actividades mencionados.
    - Conserva las recomendaciones realizadas y las decisiones tomadas.
    - Mantén los datos necesarios para comprender futuras preguntas.
    - Elimina saludos, repeticiones e información irrelevante.
    - No inventes información ni añadas conocimientos externos.
    - No incluyas información que no esté presente en el historial.
    - Redacta un resumen breve, claro y estructurado.                    
    - Devuelve únicamente el texto del resumen, sin explicaciones adicionales.
"""

## Control de historial

In [75]:
MAX_TOKENS = 5000    

def exceeds_token_limit(conversation: list[types.Content]) -> bool:
    """True si se ha sobrepasado el límite de tokens establecido"""
    token_count = client.models.count_tokens(
        model=GENERATION_MODEL,
        contents=conversation
    )
    if token_count.total_tokens > MAX_TOKENS:
        return True
    else:
        return False

def conversational_context_control(conversation: list[types.Content], query: str = None) -> list[types.Content]:
    """Resume el historial si sobrepasa el límite de tokens, devuelve historial resumido"""
    if exceeds_token_limit(conversation):
        # llamada a modelo para que resuma el historial
        summary = client.models.generate_content(
            model=GENERATION_MODEL,
            contents=conversation[:-1], # excluimos última pregunta del user
            config=types.GenerateContentConfig(
                system_instruction=SUMMARIZE_LLM_SYSTEM_INSTRUCTIONS
            )
        )
        conversation: list[types.Content] = [] 
        conversation.extend(
            [
                types.Content(
                    role = "user",
                    parts=[types.Part.from_text(text=f"Resumen de la conversación anterior:\n{summary.text}")]
                ),
                types.Content(
                        role = "user",
                        parts=[types.Part.from_text(text=query)]
                )
            ]
        )
    return conversation

    

In [76]:
def run_llm_with_tools(user_input: str, conversation: list[types.Content], max_tool_calls: int = 5) -> list[dict[str, Any]]:
    """Invoca al modelo con herramientas; devulve respuesta generada, herramientas utilizadas y contexto"""
    
    # control del historial y tokens
    conversation.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_input)]
        )
    )
    conversation_and_query = conversational_context_control(conversation, user_input)
    
    # llamada al cliente con tools y ejecución automática de las mismas
    stream = client.models.generate_content_stream(
        model=GENERATION_MODEL,
        contents=conversation_and_query,
        config=types.GenerateContentConfig(
            system_instruction=MAIN_LLM_SYSTEM_INSTRUCTIONS,
            tools=[search_internal_docs_tool, get_westher_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                maximum_remote_calls=max_tool_calls
            )
        )
    )

    response = ""
    for part in stream:
        if part.text:
            print(part.text, end = "", flush=True)
            response += part.text
    print()

    # añadimos la respuesta
    conversation_and_query.append(
        types.Content(
            role="model",
            parts=[types.Part.from_text(text=response)]
        )
    )
    
    return {
        "text": response,
        "conversation": conversation_and_query
    }
    

In [77]:
# "main"
conversation: list[types.Content] = []

while True:
    query = input("Pregunta a gemini, 'exit' para salir")
    if query.lower() in ("exit", "salir"):
        break

    print("-"*80)
    print("RESPUESTA DEL MODELO")
    print()
    response = run_llm_with_tools(
        user_input = query,
        conversation = conversation
    )
    print("-"*80)
    print()
    conversation = response["conversation"]
    

Pregunta a gemini, 'exit' para salir dame el tiempo de mañama 


--------------------------------------------------------------------------------
RESPUESTA DEL MODELO

Ha ocurrido un error al consultar la predicción meteorológica para mañana. ¿Podrías indicarme la fecha exacta (en formato aaaa-mm-dd) para la que deseas consultar el tiempo?
--------------------------------------------------------------------------------



Pregunta a gemini, 'exit' para salir exit


In [57]:
for x in tool_logs:
    print(x)
    print()
    print()

{'tool': 'search_internal_docs_tool', 'arguments': {'query': 'playas recomendadas tenerife', 'k': 3}, 'success': True, 'result': [{'rank': 1, 'score': 0.765241265296936, 'chunk_id': 31, 'page': 23, 'text': '• El Médano / La Tejita \nPlayas tipiquísimas de Tenerife y otra de las estampas de la isla con Montaña Roja al \nfondo. Nota: La Tejita es playa nudista para que lo tengáis en cuenta.  \n \n \n• Los Cristianos \nDe la zona de Los Cristianos que me venga a la mente os recomendaría la Playa de \nLos Cristianos o la Playa de Las Vistas y creo que poco más (la imagen es la Playa de \nLas Vistas).'}, {'rank': 2, 'score': 0.7511739730834961, 'chunk_id': 12, 'page': 11, 'text': 'tendréis poca playa (especialmente a la del Ancón) y tened mucho respeto al mar en \nestas playas, que la corriente es muy traicionera. \n \n• Puerto de La Cruz \nEs la zona turística del Norte de Tenerife. De hecho, hace años era el núcleo turístico \nde la isla hasta que empezó a ganar mucho peso el Sur de Tener